## Sección 2: Calidad del 11 Inicial y Contexto del Partido
Las estadísticas de rendimiento no explican todo el fútbol; el talento individual y las decisiones tácticas son fundamentales. En esta sección se extrae la información contextual de **Transfermarkt**.

**Proceso de Fusión:**
1. **Filtrado:** Nos quedamos exclusivamente con partidos de `ES1` (LaLiga) desde la temporada 2019.
2. **El 11 Titular:** Cruzamos la tabla de alineaciones con las características de los jugadores para obtener la edad media y la altura.
3. **Valoración Financiera:** Mediante la función `merge_asof` (dirección *backward*), buscamos la tasación económica más reciente de cada titular **justo antes** del pitido inicial, obteniendo el `xi_market_value`.
4. **Contexto:** Añadimos el factor campo (ocupación del estadio), el nombre del entrenador y la formación táctica empleada.

In [36]:
import pandas as pd
import numpy as np
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [37]:

df_lineups = pd.read_csv('/content/drive/MyDrive/football_data/game_lineups.csv')
df_clubs = pd.read_csv('/content/drive/MyDrive/football_data/clubs.csv')
df_players = pd.read_csv('/content/drive/MyDrive/football_data/players.csv')
df_player_valuations = pd.read_csv('/content/drive/MyDrive/football_data/player_valuations.csv')
df_games = pd.read_csv('/content/drive/MyDrive/football_data/games.csv')
df_appearances = pd.read_csv('/content/drive/MyDrive/football_data/appearances.csv')
df_game_events = pd.read_csv('/content/drive/MyDrive/football_data/game_events.csv')
df_final_procesado = pd.read_csv('/content/drive/MyDrive/football_data/df_stats_procesado.csv')

/tmp/ipykernel_7913/3752038175.py:1: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df_lineups = pd.read_csv('/content/drive/MyDrive/football_data/game_lineups.csv')


In [46]:

print('lineups.csv', df_lineups.columns)
print('clubs.csv', df_clubs.columns)
print('players.csv', df_players.columns)
print('player_valuations.csv', df_player_valuations.columns)
print('games.csv', df_games.columns)
print('appearances.csv', df_appearances.columns)
print('game_events.csv', df_game_events.columns)
df_players[ (df_players['current_club_id'] == 131)]

lineups.csv Index(['game_lineups_id', 'date', 'game_id', 'player_id', 'club_id',
       'player_name', 'type', 'position', 'number', 'team_captain'],
      dtype='object')
clubs.csv Index(['club_id', 'club_code', 'name', 'domestic_competition_id',
       'total_market_value', 'squad_size', 'average_age', 'foreigners_number',
       'foreigners_percentage', 'national_team_players', 'stadium_name',
       'stadium_seats', 'net_transfer_record', 'coach_name', 'last_season',
       'filename', 'url'],
      dtype='object')
players.csv Index(['player_id', 'first_name', 'last_name', 'name', 'last_season',
       'current_club_id', 'player_code', 'country_of_birth', 'city_of_birth',
       'country_of_citizenship', 'date_of_birth', 'sub_position', 'position',
       'foot', 'height_in_cm', 'contract_expiration_date', 'agent_name',
       'image_url', 'international_caps', 'international_goals',
       'current_national_team_id', 'url',
       'current_club_domestic_competition_id', 'current_c

,player_id,first_name,last_name,name,last_season,current_club_id,player_code,country_of_birth,city_of_birth,country_of_citizenship,...,agent_name,image_url,international_caps,international_goals,current_national_team_id,url,current_club_domestic_competition_id,current_club_name,market_value_in_eur,highest_market_value_in_eur
687,7537,José Manuel,Pinto,José Manuel Pinto,2013,131,jose-manuel-pinto,Spain,El Puerto de Santa María,Spain,...,NaN,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/jose-manuel-pi...,ES1,Futbol Club Barcelona,300000.0,2000000.0
695,7594,Carles,Puyol,Carles Puyol,2013,131,carles-puyol,Spain,La Pobla de Segur,Spain,...,NaN,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/carles-puyol/p...,ES1,Futbol Club Barcelona,1000000.0,30000000.0
696,7600,Andrés,Iniesta,Andrés Iniesta,2017,131,andres-iniesta,Spain,Fuentealbilla,Spain,...,Sports&Life,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/andres-iniesta...,ES1,Futbol Club Barcelona,800000.0,70000000.0
698,7607,NaN,Xavi,Xavi,2014,131,xavi,Spain,Terrassa,Spain,...,AC Talent,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/xavi/profil/sp...,ES1,Futbol Club Barcelona,2000000.0,65000000.0
1357,15904,Thomas,Vermaelen,Thomas Vermaelen,2018,131,thomas-vermaelen,Belgium,Kapellen,Belgium,...,NaN,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/thomas-vermael...,ES1,Futbol Club Barcelona,700000.0,18500000.0
1363,15951,Dani,Alves,Dani Alves,2021,131,dani-alves,Brazil,Juazeiro,Brazil,...,"Flashforward, S.L.",https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/dani-alves/pro...,ES1,Futbol Club Barcelona,1000000.0,36000000.0
1590,18944,Gerard,Piqué,Gerard Piqué,2022,131,gerard-pique,Spain,Barcelona,Spain,...,AC Talent,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/gerard-pique/p...,ES1,Futbol Club Barcelona,3000000.0,50000000.0
1676,19981,Javier,Mascherano,Javier Mascherano,2017,131,javier-mascherano,Argentina,San Lorenzo,Argentina,...,NaN,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/javier-mascher...,ES1,Futbol Club Barcelona,325000.0,30000000.0
2231,26399,Sergio,Agüero,Sergio Agüero,2021,131,sergio-aguero,Argentina,Quilmes,Argentina,...,Eleven Talent Group,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/sergio-aguero/...,ES1,Futbol Club Barcelona,15000000.0,80000000.0
3273,38253,Robert,Lewandowski,Robert Lewandowski,2025,131,robert-lewandowski,Poland,Warszawa,Poland,...,Gol International,https://img.a.transfermarkt.technology/portrai...,164.0,89.0,NaN,https://www.transfermarkt.co.uk/robert-lewando...,ES1,Futbol Club Barcelona,8000000.0,90000000.0


## Filtrado por temporada

In [47]:
import pandas as pd
import numpy as np


# 1. FILTRADO INICIAL (La Liga desde 2019)


df_games_liga = df_games[
    (df_games['competition_id'] == 'ES1') &
    (df_games['season'] >= 2019)
].copy()

juegos_liga_ids = df_games_liga['game_id'].unique()

df_lineups_liga = df_lineups[
    (df_lineups['game_id'].isin(juegos_liga_ids)) &
    (df_lineups['type'] == 'starting_lineup')
].copy()


# 2. ENRIQUECIMIENTO CON DATOS DEL JUGADOR

# Cruzamos con la tabla players para traer nacimiento, altura e internacionalidades
df_titulares = df_lineups_liga.merge(
    df_players[['player_id', 'date_of_birth', 'height_in_cm', 'international_caps']],
    on='player_id',
    how='left'
)


# Convertimos a formato fecha.

df_titulares['date'] = pd.to_datetime(df_titulares['date'])
df_titulares['date_of_birth'] = pd.to_datetime(df_titulares['date_of_birth'])

# Calculamos la edad exacta del jugador el día de ese partido
df_titulares['edad_partido'] = (df_titulares['date'] - df_titulares['date_of_birth']).dt.days / 365.25



# CALCULO DE VALOR DE MERCADO HISTÓRICO

df_player_valuations['date'] = pd.to_datetime(df_player_valuations['date'])

# Ordenamos por fecha para que funcione el merge_asof
df_valuations_sorted = df_player_valuations.sort_values('date')
df_titulares = df_titulares.sort_values('date')

# Buscamos la tasación más cercana ANTERIOR al partido
df_titulares_valor = pd.merge_asof(
    df_titulares,
    df_valuations_sorted[['player_id', 'date', 'market_value_in_eur']],
    on='date',
    by='player_id',
    direction='backward'
)


# AGREGACIÓN (1 Fila por Equipo y Partido)

df_calidad_11 = df_titulares_valor.groupby(['game_id', 'club_id','date']).agg(
    xi_market_value=('market_value_in_eur', 'sum'), # Valor total del 11 titular
    xi_average_age=('edad_partido', 'mean'),        # Edad media
    xi_average_height=('height_in_cm', 'mean'),     # Altura media
    xi_total_caps=('international_caps', 'sum')     # Experiencia total en selecciones
).reset_index()

# Rellenamos posibles NaNs
df_calidad_11['xi_market_value'] = df_calidad_11['xi_market_value'].fillna(0)
df_calidad_11['xi_average_height'] = df_calidad_11['xi_average_height'].fillna(df_calidad_11['xi_average_height'].mean())
df_calidad_11['xi_total_caps'] = df_calidad_11['xi_total_caps'].fillna(0)

print(f"Filas generadas: {len(df_calidad_11)}")
df_calidad_11.columns

Filas generadas: 5131


Index(['game_id', 'club_id', 'date', 'xi_market_value', 'xi_average_age',
       'xi_average_height', 'xi_total_caps'],
      dtype='object')

In [48]:

# PASAMOS DATOS DE GAMES A FORMATO VERTICAL

# Extraemos la perspectiva del equipo LOCAL
df_home = df_games_liga[[
    'game_id', 'home_club_id', 'home_club_formation',
    'home_club_manager_name', 'attendance'
]].copy()

# Renombramos para estandarizar
df_home.columns = ['game_id', 'club_id', 'formation', 'manager_name', 'attendance']
df_home['es_local_context'] = 1

# Extraemos la perspectiva del equipo VISITANTE
df_away = df_games_liga[[
    'game_id', 'away_club_id', 'away_club_formation',
    'away_club_manager_name', 'attendance'
]].copy()


# Renombramos para estandarizar
df_away.columns = ['game_id', 'club_id', 'formation', 'manager_name', 'attendance']
df_away['es_local_context'] = 0

# Unimos ambos (Ahora tenemos 1 fila por equipo y partido, igual que df_calidad_11)
df_contexto = pd.concat([df_home, df_away], ignore_index=True)


# EL FACTOR ESTADIO (Ocupación y Presión)

# Traemos la capacidad del estadio desde la tabla clubs
df_contexto = df_contexto.merge(
    df_clubs[['club_id', 'stadium_seats', 'name']],
    on='club_id',
    how='left'
)

# Calculamos el % de ocupación (Attendance / Capacidad)
# Ojo: la capacidad pertenece al club, por lo que el visitante también verá
# la capacidad del estadio al que viaja en la misma fila del partido.
# Rellenamos divisiones por cero o valores vacíos con 0
df_contexto['ocupacion_estadio'] = df_contexto['attendance'] / df_contexto['stadium_seats'].replace(0, np.nan)
df_contexto['ocupacion_estadio'] = df_contexto['ocupacion_estadio'].fillna(0).clip(upper=1.0) # Evitar > 100% por errores



# 3. UNIÓN FINAL DEL NUEVO DATASET

# Juntamos la tabla de Calidad (xi_market_value...) con la de Contexto
df_calidad_contexto = df_calidad_11.merge(
    df_contexto[['game_id', 'club_id', 'formation', 'manager_name', 'ocupacion_estadio']],
    on=['game_id', 'club_id'],
    how='inner' # Inner para quedarnos solo con cruces perfectos
)

# Limpieza rápida de strings (para que la Red Neuronal no se líe con mayúsculas)
df_calidad_contexto['formation'] = df_calidad_contexto['formation'].astype(str).str.strip().str.upper()
df_calidad_contexto['manager_name'] = df_calidad_contexto['manager_name'].astype(str).str.strip().str.lower()

print(f"Filas tras el merge: {len(df_calidad_contexto)}")
print(df_calidad_contexto[['xi_market_value', 'formation', 'manager_name', 'ocupacion_estadio']].head())
df_calidad_contexto.columns

Filas tras el merge: 5131
   xi_market_value        formation          manager_name  ocupacion_estadio
0      510000000.0  4-3-3 ATTACKING       zinédine zidane           0.283293
1      121000000.0            4-4-2          fran escribá           0.947567
2       14800000.0          4-3-1-2        vicente moreno           0.581360
3       42100000.0   4-4-2 DOUBLE 6  josé luis mendilibar           1.000000
4      614000000.0   4-4-2 DOUBLE 6         diego simeone           0.781990


Index(['game_id', 'club_id', 'date', 'xi_market_value', 'xi_average_age',
       'xi_average_height', 'xi_total_caps', 'formation', 'manager_name',
       'ocupacion_estadio'],
      dtype='object')

In [49]:
df_clubs[df_clubs['club_id'].isin(df_calidad_11['club_id'].values)][['club_id','club_code']].values
pd.unique(df_final_procesado.Equipo)

array(['Alaves', 'Almeria', 'Athletic Club', 'Atletico Madrid',
       'Barcelona', 'Cadiz', 'Celta Vigo', 'Eibar', 'Elche', 'Espanyol',
       'Getafe', 'Girona', 'Granada', 'Las Palmas', 'Leganes', 'Levante',
       'Mallorca', 'Osasuna', 'Rayo Vallecano', 'Real Betis',
       'Real Madrid', 'Real Oviedo', 'Real Sociedad', 'Real Valladolid',
       'SD Huesca', 'Sevilla', 'Valencia', 'Villarreal'], dtype=object)

In [50]:
team_id_mapping = {
    'Alaves': 1108,
    'Almeria': 3302,
    'Athletic Club': 621,
    'Atletico Madrid': 13,
    'Barcelona': 131,
    'Cadiz': 2687,
    'Celta Vigo': 940,
    'Eibar': 1533,
    'Elche': 1531,
    'Espanyol': 714,
    'Getafe': 3709,
    'Girona': 12321,
    'Granada': 16795,
    'Las Palmas': 472,
    'Leganes': 1244,
    'Levante': 3368,
    'Mallorca': 237,
    'Osasuna': 331,
    'Rayo Vallecano': 367,
    'Real Betis': 150,
    'Real Madrid': 418,
    'Real Oviedo': 2497,
    'Real Sociedad': 681,
    'Real Valladolid': 366,
    'SD Huesca': 5358,
    'Sevilla': 368,
    'Valencia': 1049,
    'Villarreal': 1050
}
# APLICAMOS EL MAPEO
df_final_procesado['club_id'] = df_final_procesado['Equipo'].map(team_id_mapping)

# Verificamos si hay algún equipo sin mapear (Saldría como NaN)
equipos_sin_mapear = df_final_procesado[df_final_procesado['club_id'].isna()]['Equipo'].unique()
print(f"Equipos sin mapear: {equipos_sin_mapear}")
# Si sale vacío [], ¡perfecto! Si sale alguno, añádelo al diccionario.

Equipos sin mapear: []


In [51]:
# Asegurar formatos de fecha
df_final_procesado['Fecha'] = pd.to_datetime(df_final_procesado['Fecha'])
df_calidad_contexto['date'] = pd.to_datetime(df_calidad_contexto['date'])


# Hacemos el Merge con el dataframe de las stats avg5 calculadas anteriormente
df_modelo_base = pd.merge(
    df_final_procesado,
    df_calidad_contexto,
    left_on=['club_id', 'Fecha'],
    right_on=['club_id', 'date'],
    how='inner'
)



# Limpiamos columnas redundantes
df_modelo_base = df_modelo_base.drop(columns=['date'])
print(f"Filas tras el merge: {len(df_modelo_base)}")
df_modelo_base.to_csv('df_completo_por_equipos', index=False)
df_modelo_base.columns

Filas tras el merge: 4367


Index(['Equipo', 'Fecha', 'hora', 'es_local', 'resultado_final', 'xpts_avg5',
       'ppda_avg5', 'corners_avg5', 'faltas_avg5', 'amarillas_avg5',
       'rojas_avg5', 'diff_xg_avg5', 'diff_tiros_puerta_avg5',
       'diff_tiros_avg5', 'diff_pases_area_avg5', 'diff_xpts_avg5',
       'diff_corners_avg5', 'diff_faltas_avg5', 'diff_amarillas_avg5',
       'diff_ratio_presion_avg5', 'diff_dominio_ofensivo_avg5',
       'diff_agresividad_ofensiva_avg5', 'diff_disciplina_avg5', 'xpts_ewma5',
       'ppda_ewma5', 'corners_ewma5', 'faltas_ewma5', 'amarillas_ewma5',
       'rojas_ewma5', 'diff_xg_ewma5', 'diff_tiros_puerta_ewma5',
       'diff_tiros_ewma5', 'diff_pases_area_ewma5', 'diff_xpts_ewma5',
       'diff_corners_ewma5', 'diff_faltas_ewma5', 'diff_amarillas_ewma5',
       'diff_ratio_presion_ewma5', 'diff_dominio_ofensivo_ewma5',
       'diff_agresividad_ofensiva_ewma5', 'diff_disciplina_ewma5', 'club_id',
       'game_id', 'xi_market_value', 'xi_average_age', 'xi_average_height',
    

In [52]:
# 1. Separamos Locales y Visitantes
df_locales = df_modelo_base[df_modelo_base['es_local'] == 1].copy()
df_visitantes = df_modelo_base[df_modelo_base['es_local'] == 0].copy()

# 2. Añadimos sufijos para no mezclar columnas (_L para Local, _V para Visitante)
# Mantenemos 'game_id', 'Fecha' y 'resultado_final' sin sufijo para usarlos de llave
cols_identificadoras = ['game_id', 'Fecha', 'resultado_final','hora']

df_locales = df_locales.set_index(cols_identificadoras).add_suffix('_L').reset_index()
df_visitantes = df_visitantes.set_index(cols_identificadoras).add_suffix('_V').reset_index()

# OJO: Quitamos el 'resultado_final' del visitante para que no se duplique
df_visitantes = df_visitantes.drop(columns=['resultado_final'])

# 3. MERGE FINAL POR PARTIDO
df_partidos = pd.merge(
    df_locales,
    df_visitantes,
    on=['game_id', 'Fecha','hora'],
    how='inner'
)

# Limpiamos variables inútiles como 'es_local_L' (siempre será 1) o 'es_local_V' (siempre será 0)
columnas_a_borrar = [col for col in df_partidos.columns if 'es_local' in col]
df_partidos = df_partidos.drop(columns=columnas_a_borrar)
df_partidos.columns
print(f"Total de partidos listos para la Red Neuronal: {len(df_partidos)}")

Total de partidos listos para la Red Neuronal: 2183


In [53]:
df_partidos.to_csv('df_completo_modelo', index=False)
df_partidos.columns

Index(['game_id', 'Fecha', 'resultado_final', 'hora', 'Equipo_L',
       'xpts_avg5_L', 'ppda_avg5_L', 'corners_avg5_L', 'faltas_avg5_L',
       'amarillas_avg5_L', 'rojas_avg5_L', 'diff_xg_avg5_L',
       'diff_tiros_puerta_avg5_L', 'diff_tiros_avg5_L',
       'diff_pases_area_avg5_L', 'diff_xpts_avg5_L', 'diff_corners_avg5_L',
       'diff_faltas_avg5_L', 'diff_amarillas_avg5_L',
       'diff_ratio_presion_avg5_L', 'diff_dominio_ofensivo_avg5_L',
       'diff_agresividad_ofensiva_avg5_L', 'diff_disciplina_avg5_L',
       'xpts_ewma5_L', 'ppda_ewma5_L', 'corners_ewma5_L', 'faltas_ewma5_L',
       'amarillas_ewma5_L', 'rojas_ewma5_L', 'diff_xg_ewma5_L',
       'diff_tiros_puerta_ewma5_L', 'diff_tiros_ewma5_L',
       'diff_pases_area_ewma5_L', 'diff_xpts_ewma5_L', 'diff_corners_ewma5_L',
       'diff_faltas_ewma5_L', 'diff_amarillas_ewma5_L',
       'diff_ratio_presion_ewma5_L', 'diff_dominio_ofensivo_ewma5_L',
       'diff_agresividad_ofensiva_ewma5_L', 'diff_disciplina_ewma5_L',
    